### Transformar Datos de "Departments"
1. Eliminar registros NULL del campo "department_id"
2. Eliminar registros duplicados "identicos"
3. Eliminar registros duplicados en base al campo "created_timestamp"
4. Convertir las columnas al "tipo de dato" correcto
5. Escribir los datos "transformados" en el schema "silver"

#### 1. Eliminar registros NULL del campo "department_id"

In [0]:
departments_df = spark.table("zenviro.bronze.departments_py")
#Notación de objetos
filtered_df = departments_df.filter(departments_df.department_id.isNotNull()).orderBy("department_id")
display(filtered_df)

#### 2. Eliminar registros duplicados "identicos"

In [0]:
distinct_df = filtered_df.dropDuplicates().orderBy("department_id")
display(distinct_df)

In [0]:
from pyspark.sql.functions import max

max_ts_df = distinct_df.groupBy("department_id") \
            .agg(max('created_timestamp').alias("max_created_timestamp"))
display(max_ts_df)

#### 3. Eliminar registros duplicados en base al campo "created_timestamp"

In [0]:
distinct_ct_df = (
  distinct_df
  .join(max_ts_df, 
        (distinct_df.department_id == max_ts_df.department_id) & 
        (distinct_df.created_timestamp == max_ts_df.max_created_timestamp),
        "inner")
  .select(distinct_df["*"])
)
display(distinct_ct_df)

#### 4. Convertir las columnas al "tipo de dato" correcto

In [0]:
casted_df = (
    distinct_ct_df
    .select(distinct_ct_df.created_timestamp.cast("timestamp"),
            distinct_ct_df.department_id,
            distinct_ct_df.location_id,
            distinct_ct_df.department_name)
)
display(casted_df)

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
casted_df.writeTo("zenviro.silver.departments_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.departments_py"))